In [44]:
import os
import json
import h5py
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Video
import imageio
from statistics import mean
import robomimic.utils.file_utils as FileUtils
from PIL import Image, ImageDraw, ImageFont
import zarr
from diffusion_policy.common.replay_buffer import ReplayBuffer
from filelock import FileLock
from diffusion_policy.codecs.imagecodecs_numcodecs import register_codecs, Jpeg2k
import pdb
from tqdm import tqdm
import xml.etree.ElementTree as ET


In [159]:
og_redcube_data = h5py.File('/proj/vondrick3/sruthi/robots/diffusion_policy/data/robomimic/datasets/lift/ph/robomimic/datasets/lift/ph/image_abs.hdf5', 'r')
# print(og_redcube_data['data'].attrs['env_args'])

# Create Saved Rollout Dataset

In [176]:
temp = np.load('/proj/vondrick3/sruthi/robots/diffusion_policy/data/outputs/2024.06.05/15.00.33_train_diffusion_unet_hybrid_liftph/checkpoints/epoch=0150-test_mean_score=0.980/alift_25predcube_11_21_4_14/obsdict_eyeinhand.npy')
print('hey', temp.shape)

temp = np.load('/proj/vondrick3/sruthi/robots/diffusion_policy/data/outputs/2024.06.05/15.00.33_train_diffusion_unet_hybrid_liftph/checkpoints/epoch=0150-test_mean_score=0.980/alift_25predcube_11_21_4_14/obsdict_robot0s.npy')
print('hello', temp.shape)

temp = np.load('/proj/vondrick3/sruthi/robots/diffusion_policy/data/outputs/2024.06.05/15.00.33_train_diffusion_unet_hybrid_liftph/checkpoints/epoch=0150-test_mean_score=0.980/alift_25predcube_11_21_4_14/actions.npy')
print('hi', temp.shape)

temp = np.load('/proj/vondrick3/sruthi/robots/diffusion_policy/data/outputs/2024.06.05/15.00.33_train_diffusion_unet_hybrid_liftph/checkpoints/epoch=0150-test_mean_score=0.980/alift_25predcube_11_21_4_14/rewards.npy')
print('hello', temp.shape)

temp = np.load('/proj/vondrick3/sruthi/robots/diffusion_policy/data/outputs/2024.06.05/15.00.33_train_diffusion_unet_hybrid_liftph/checkpoints/epoch=0150-test_mean_score=0.980/alift_25predcube_11_21_4_14/grasping.npy')
print('hello', temp.shape)

temp = np.load('/proj/vondrick3/sruthi/robots/diffusion_policy/data/outputs/2024.06.05/15.00.33_train_diffusion_unet_hybrid_liftph/checkpoints/epoch=0150-test_mean_score=0.980/alift_25predcube_11_21_4_14/startstates.npy')
print('hello', temp.shape)

hey (13, 1008, 8, 3, 84, 84)
hello (13, 1008, 8, 9)
hi (13, 1008, 8, 7)
hello (1008, 100)
hello (1008, 100)
hello (1008, 32)


In [149]:
basepath = '/proj/vondrick3/sruthi/robots/diffusion_policy/data/outputs/2025.01.13/12.45.41_train_diffusion_unet_hybrid_robocasalang_PnPSinkToCounter_trainsplit_imagenet/checkpoints/epoch=0600-val_loss=0.079/PnPSinkToCounter_None_1_13_23_24_6/'
obsdict_robot0_agentview_right_image = np.load(basepath + 'obsdict_robot0_agentview_right_image.npy', 'r')
obsdict_robot0_agentview_left_image = np.load(basepath + 'obsdict_robot0_agentview_left_image.npy', 'r')
obsdict_eyeinhand = np.load(basepath + 'obsdict_eyeinhand.npy', 'r')
obsdict_robot0s = np.load(basepath + 'obsdict_robot0s.npy', 'r')
rewards = np.load(basepath + 'rewards.npy', 'r')
states = np.load(basepath + 'startstates.npy', 'r')
actions = np.load(basepath + 'actions.npy', 'r')
grasping = np.load(basepath + 'grasping.npy', 'r')

In [150]:
obsdict_robot0_agentview_right_image.shape

(51, 2, 2, 3, 128, 128)

In [107]:
print('hi', basepath)
obsdict_robot0_agentview_right_image = (obsdict_robot0_agentview_right_image.transpose(0,2,1,4,5,3).reshape(13*8,1008,84,84,3)* 255.0).astype(np.uint8)[:-4]
print('obsdict_robot0_agentview_right_image', obsdict_robot0_agentview_right_image.shape)

obsdict_robot0_agentview_left_image = (obsdict_robot0_agentview_left_image.transpose(0,2,1,4,5,3).reshape(13*8,1008,84,84,3)* 255.0).astype(np.uint8)[:-4]
print('obsdict_robot0_agentview_left_image', obsdict_robot0_agentview_left_image.shape)

obsdict_eyeinhand = (obsdict_eyeinhand.transpose(0,2,1,4,5,3).reshape(13*8,1008,84,84,3)* 255.0).astype(np.uint8)[:-4]
print('obsdict_eyeinhand', obsdict_eyeinhand.shape)

obsdict_robot0s = obsdict_robot0s.transpose(0,2,1,3).reshape(13*8,1008,9)[:-4]
print('obsdict_robot0s', obsdict_robot0s.shape)

actions = np.load(trial_hammer_basepath + 'actions.npy', 'r')
print(actions.shape)
actions = actions[:,:,:8,:].transpose(0,2,1,3).reshape(13*8,1008,7)[:-4]
print('actions', actions.shape)

rewards = rewards.transpose(1,0)
print('rewards', rewards.shape)

states = np.repeat(np.array(states)[np.newaxis, :, :], obsdict_agentview.shape[0], axis=0)
print('states', states.shape)

hi /proj/vondrick3/sruthi/robots/diffusion_policy/data/outputs/2024.09.03/21.23.37_train_diffusion_unet_hybrid_15.00.33_check/checkpoints/epoch=0150-test_mean_score=0.940/unguided/alift_4wredcube_11_14_23_34_15/
obsdict_agentview (100, 1008, 84, 84, 3)
obsdict_eyeinhand (100, 1008, 84, 84, 3)
obsdict_robot0s (100, 1008, 9)
(13, 1008, 8, 7)
actions (100, 1008, 7)
rewards (100, 1008)
states (100, 1008, 32)


expected:

hi
obsdict_agentview (100, 1008, 84, 84, 3)
obsdict_eyeinhand (100, 1008, 84, 84, 3)
obsdict_robot0s (100, 1008, 9)
(13, 1008, 8, 7)
actions (100, 1008, 7)
rewards (100, 1008)
states (100, 1008, 32)

In [181]:
test = h5py.File('test.hdf5', 'w')


OSError: Unable to create file (unable to truncate a file which is already open)

In [188]:
test['data'].attrs['temp']=json.dumps({})

In [102]:
''' CREATE 1 DATASET '''
new_data = {'data':{}}

count = 0
'''
#for high quality dataset
for datapt in range(0,rewards.shape[1]):
    success = int(np.max(rewards[:,datapt]))
    minsuccess = min(np.argwhere(rewards[:,datapt]==1))[0] if success else -1

    if minsuccess>0 and rewards[:,datapt][minsuccess:minsuccess+10].sum()==10:
        stop_demo = minsuccess+16
        new_data['data'][f'demo_{count}'] = {
            'rewards': rewards[:stop_demo,datapt],
            'success': [success],
            'og_pt': [datapt],
            'states': states[:stop_demo,datapt,:],
            'actions': actions[:stop_demo, datapt],
            'obs': {
                'agentview_image': obsdict_agentview[:stop_demo, datapt],
                'robot0_eye_in_hand_image': obsdict_eyeinhand[:stop_demo, datapt],
                'robot0_eef_pos': obsdict_robot0s[:stop_demo, datapt, :3],
                'robot0_eef_quat': obsdict_robot0s[:stop_demo, datapt, 3:7],
                'robot0_gripper_qpos': obsdict_robot0s[:stop_demo, datapt, 7:],
            }
        }
        count+=1 
    else:
        stop_demo = -1
'''

for datapt in range(0,rewards.shape[1]):
    success = np.max(rewards[:,datapt])
    if success>0:
        stop_demo = min(np.argwhere(rewards[:,datapt]==1))[0] + 16
        new_data['data'][f'demo_{count}'] = {
            'rewards': rewards[:stop_demo,datapt],
            'success': [success],
            'og_pt': [datapt],
            'states': states[:stop_demo,datapt,:],
            'actions': actions[:stop_demo, datapt],
            'obs': {
                'agentview_image': obsdict_agentview[:stop_demo, datapt],
                'robot0_eye_in_hand_image': obsdict_eyeinhand[:stop_demo, datapt],
                'robot0_eef_pos': obsdict_robot0s[:stop_demo, datapt, :3],
                'robot0_eef_quat': obsdict_robot0s[:stop_demo, datapt, 3:7],
                'robot0_gripper_qpos': obsdict_robot0s[:stop_demo, datapt, 7:],
            }
        }
        count+=1 
    else:
        stop_demo = -1   
    
# Open HDF5 file and write in the data_dict structure and info
savepath = trial_hammer_basepath+'data_successful_only.hdf5'
f = h5py.File(savepath, 'w')
datagrp = f.create_group('data')


datagrp.attrs['env_args'] = og_redcube_data['data'].attrs['env_args']
datagrp.attrs['total'] = len(new_data['data'])


for demo in new_data['data']:
    demogrp = datagrp.create_group(demo)
    
    actionsdset = demogrp.create_dataset('actions', data = new_data['data'][demo]['actions'])
    rewardsdset = demogrp.create_dataset('rewards', data = new_data['data'][demo]['rewards'])
    successdset = demogrp.create_dataset('success', data = new_data['data'][demo]['success'])
    statesdset = demogrp.create_dataset('states', data = new_data['data'][demo]['states'])
    statesdset = demogrp.create_dataset('og_pt', data = new_data['data'][demo]['og_pt'])

    obsgrp = demogrp.create_group('obs') 
    for grp_name in new_data['data'][demo]['obs']:
        dset = obsgrp.create_dataset(grp_name, data = new_data['data'][demo]['obs'][grp_name])
print('demo done', demo)
f.close()


demo done demo_612


In [48]:
print(trial_hammer_basepath+'data_all.hdf5')
print(h5py.File(trial_hammer_basepath+'data_alll.hdf5')['data'])
print(h5py.File(trial_hammer_basepath+'data_unsuccessful_only.hdf5')['data'])
print(h5py.File(trial_hammer_basepath+'data_successful_only.hdf5')['data'])

/proj/vondrick3/sruthi/robots/diffusion_policy/data/outputs/2024.09.03/21.23.37_train_diffusion_unet_hybrid_15.00.33_check/checkpoints/epoch=0150-test_mean_score=0.940/unguided/alift_4wredcube_11_6_11_37_51/data_all.hdf5


FileNotFoundError: [Errno 2] Unable to open file (unable to open file: name = '/proj/vondrick3/sruthi/robots/diffusion_policy/data/outputs/2024.09.03/21.23.37_train_diffusion_unet_hybrid_15.00.33_check/checkpoints/epoch=0150-test_mean_score=0.940/unguided/alift_4wredcube_11_6_11_37_51/data_alll.hdf5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [49]:
'''TEST THE NEW DATASET'''

data = h5py.File(trial_hammer_basepath + 'data_all.hdf5', 'r')
print(data['data'])
video_path = trial_hammer_basepath+'temp.mp4'
video_writer = imageio.get_writer(video_path, fps=20)
idx = 50
demo = f'demo_{idx}'
print('shape', data['data'][demo]['obs']['agentview_image'].shape)
print('success', data['data'][demo]['success'][:])
print('ogpt', data['data'][demo]['og_pt'][0])
for b in  data['data'][demo]['obs']['agentview_image']:
    img = Image.fromarray((b).astype(np.uint8))
    d = ImageDraw.Draw(img)
    d.text( (2,2), str(idx), fill=255)
    
    #-- back to array
    b = np.asarray(img)

    video_writer.append_data(b)
    idx+=1
video_writer.close()
Video(video_path, embed=True)
data.close()

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (84, 84) to (96, 96) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


<HDF5 group "/data" (1008 members)>
shape (43, 84, 84, 3)
success [1.]
ogpt 50


[swscaler @ 0x686b280] Warning: data is not aligned! This can lead to a speed loss


# Testing other datasets

In [151]:
data=h5py.File('/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPSinkToCounter/2024-04-26_2/demo_gentex_im128_randcams_new_images.hdf5','r')

In [126]:
trainli = set()
valli=set()
for demo in data['mask']['valid']:
    try:
        valli.add(json.loads(data['data'][demo].attrs['ep_meta'])['gen_textures']['counter_tex'].split('/')[-1])
    except Exception as e :
        print('val issue',demo, e)
for demo in data['mask']['train']:
    try:
        if json.loads(data['data'][demo].attrs['ep_meta'])['gen_textures']['counter_tex'].split('/')[-1] in trainli:
            print('dupe')
        trainli.add(json.loads(data['data'][demo].attrs['ep_meta'])['gen_textures']['counter_tex'].split('/')[-1])
    except Exception as e:
        print('train issue',demo, e)
print(len(data['data']))
print(len(trainli), trainli)
print()
print(len(valli),valli)

dupe
dupe
dupe
dupe
dupe
dupe
dupe
dupe
train issue b'demo_53' "Unable to open object (object 'demo_53' doesn't exist)"
train issue b'demo_54' "Unable to open object (object 'demo_54' doesn't exist)"
train issue b'demo_55' "Unable to open object (object 'demo_55' doesn't exist)"
dupe
53
38 {'c9d714cd-b00b-4d96-b6a6-cdc9e01c1b71.png', '9c47a3ac-2e56-47c1-86d4-006f98e21987.png', '827976d2-dbab-47a3-905f-602e945a6b30.png', '014ac251-0435-41c2-9dee-c664ee268327.png', '2408cfed-ad91-43e2-9295-fdafafb76c8d.png', '9e1fd142-1d1c-4a82-a961-681db31b343f.png', '1c72f7db-374f-47db-8424-9526d4e7fa7a.png', '1b23c19f-7e79-4b96-a744-42615a72d722.png', '1ab263d2-ea24-4fe7-9383-f227aadc8c47.png', 'f1299a54-59e5-42ea-9f53-e9f11fd6ca3c.png', 'fe8afe85-5153-48cb-80e0-88de2a577f0c.png', '8c6ed656-6d1c-4707-82a2-cbff87a589f7.png', 'adcb6ab3-7ecf-47f9-a11e-6a505ad23d98.png', '932f5612-9058-4923-820d-894d630190cb.png', '08ca0b3f-17e9-476e-9c17-ff81616c6bd7.png', '25d4c91a-c831-4249-b78e-a4b25d0b1ea1.png', '561

In [95]:
json.loads(data['data'].attrs['env_args'])

{'env_name': 'PnPSinkToCounter',
 'env_version': '1.5.0',
 'type': 1,
 'env_kwargs': {'robots': 'PandaMobile',
  'controller_configs': {'type': 'OSC_POSE',
   'input_max': 1,
   'input_min': -1,
   'output_max': [0.05, 0.05, 0.05, 0.5, 0.5, 0.5],
   'output_min': [-0.05, -0.05, -0.05, -0.5, -0.5, -0.5],
   'kp': 150,
   'damping_ratio': 1,
   'impedance_mode': 'fixed',
   'kp_limits': [0, 300],
   'damping_ratio_limits': [0, 10],
   'position_limits': None,
   'orientation_limits': None,
   'uncouple_pos_ori': True,
   'control_delta': True,
   'interpolation': None,
   'ramp_ratio': 0.2},
  'layout_ids': -1,
  'style_ids': [0, 1, 2, 3, 4, 5, 6, 7, 8, 11],
  'translucent_robot': False,
  'obj_instance_split': 'A',
  'generative_textures': '100p',
  'randomize_cameras': True,
  'camera_heights': 128,
  'camera_widths': 128,
  'has_renderer': False,
  'has_offscreen_renderer': True,
  'ignore_done': True,
  'use_object_obs': True,
  'use_camera_obs': True,
  'camera_depths': False,
  're

In [109]:
json.loads(data['data']['demo_0'].attrs['ep_meta'])['gen_textures']



{'cab_tex': '/proj/vondrick3/sruthi/robots/robocasa/robocasa/models/assets/generative_textures/cabinet/flat copy 26.png',
 'counter_tex': '/proj/vondrick3/sruthi/robots/robocasa/robocasa/models/assets/generative_textures/counter/5065776e-1fbc-4a70-a786-9a2ec6ee5aa0.png',
 'floor_tex': '/proj/vondrick3/sruthi/robots/robocasa/robocasa/models/assets/generative_textures/floor/8e4ea598-7170-4d24-ad12-42d45ef17a3b.png',
 'wall_tex': '/proj/vondrick3/sruthi/robots/robocasa/robocasa/models/assets/generative_textures/wall/plain copy 49.png'}

In [24]:
for root, _, files in os.walk('/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage'):
    for file in files:
        if file=='demo_gentex_im128_randcams_new_images.hdf5':
            data=h5py.File(os.path.join(root, file), 'r')
            all_objects = {}
            # train_objects = set()
            # valid_objects = set()
            for x in data['data']:
                try:
                    for objs in json.loads(data['data'][x].attrs['ep_meta'])['object_cfgs']:
                        if objs['name']=='obj':
                            all_objects[objs['info']['cat']] = all_objects.get(objs['info']['cat'],0) + 1
                except:
                    print('issue:', root, x)
            # for x in data['mask']['valid']:
            #     try:
            #         for objs in json.loads(data['data'][x].attrs['ep_meta'])['object_cfgs']:
            #             if objs['name']=='obj':
            #                 if objs['info']['cat'] in valid_objects:
            #                     print('valid seeing this again', objs['info']['cat'])
            #                 valid_objects.add(objs['info']['cat'])
            #     except Exception as e:
            #         print('valid huh,', e)
            # for x in data['mask']['train']:
            #     try:
            #         for objs in json.loads(data['data'][x].attrs['ep_meta'])['object_cfgs']:
            #             if objs['name']=='obj':
            #                 if objs['info']['cat'] in valid_objects:
            #                     print('THIS IS IN VALID', objs['info']['cat'])
            #                 if objs['info']['cat'] in train_objects:
            #                     print('train seeing this again', objs['info']['cat'])
            #                 train_objects.add(objs['info']['cat'])
            #     except Exception as e:
            #         pass #print()#'train huh,', e)
            all_objects = dict(sorted(all_objects.items(), key=lambda item: item[1]))
            print('DONE', len(all_objects), root.split('/')[-2], all_objects)

DONE 19 PnPStoveToCounter {'onion': 1, 'broccoli': 1, 'fish': 1, 'bell_pepper': 1, 'carrot': 2, 'eggplant': 2, 'garlic': 2, 'apple': 2, 'sweet_potato': 3, 'lime': 3, 'cheese': 3, 'potato': 3, 'mushroom': 3, 'egg': 3, 'lemon': 4, 'tomato': 4, 'steak': 5, 'squash': 5, 'mango': 5}
DONE 27 PnPSinkToCounter {'avocado': 1, 'kiwi': 1, 'bell_pepper': 1, 'corn': 1, 'tangerine': 1, 'lemon': 1, 'squash': 1, 'steak': 1, 'egg': 1, 'sweet_potato': 1, 'orange': 1, 'peach': 1, 'potato': 2, 'mushroom': 2, 'fish': 2, 'banana': 2, 'broccoli': 2, 'milk': 2, 'mango': 2, 'garlic': 3, 'onion': 3, 'apple': 3, 'pear': 3, 'yogurt': 3, 'lime': 3, 'carrot': 4, 'tomato': 5}
DONE 17 PnPMicrowaveToCounter {'tomato': 1, 'bell_pepper': 1, 'mushroom': 1, 'cheese': 2, 'sweet_potato': 2, 'steak': 2, 'garlic': 2, 'eggplant': 3, 'hot_dog': 3, 'fish': 3, 'onion': 4, 'egg': 4, 'corn': 4, 'potato': 4, 'squash': 5, 'carrot': 5, 'broccoli': 6}
DONE 18 PnPCounterToStove {'steak': 1, 'egg': 1, 'lemon': 1, 'eggplant': 1, 'mango': 

In [ ]:
path='/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPStoveToCounter/2024-05-01/demo_gentex_im128_randcams_new_images.hdf5'
data=h5py.File(path,'r')
demo_keys = sorted(data['data'], key=lambda x: int(x.split('_')[1]))
ep_lens = []
langset = []
for demo in demo_keys:
    ep_lens.append(data['data'][demo]['actions'].shape[0])
    max_ep=max(max_ep,data['data'][demo]['actions'].shape[0])
    langset.append(demo+': '+json.loads(data['data'][demo].attrs['ep_meta'])['lang'].split('from the pan and place it on the plate')[0])
print('len',len(ep_lens))
print('max',max(ep_lens))
print('avg',mean(ep_lens))
print('langset',*langset, sep="\n")

In [ ]:
basepath='/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/combined/PnPXToCounter/demo_gentex_im128_randcams_new_images.hdf5'
data=h5py.File(f'{basepath}', 'r')

for side in ['agentview_left','agentview_right','eye_in_hand']:
    if not os.path.exists(basepath + f'/demos/{side}'):
        os.makedirs(basepath + f'/demos/{side}')
    vidcount=0
    for demo in data['data']:
        if vidcount < 11:
            video_path = f'{basepath}/demos/{side}/{demo}.mp4'
            video_writer = imageio.get_writer(video_path, fps=20)
            if 'success' in data['data'][demo]:
                print('success',data['data'][demo]['success'][0])
            idx=0
            for b in  data['data'][demo]['obs']['robot0_'+side+'_image']:
                img = Image.fromarray((b).astype(np.uint8))
                d = ImageDraw.Draw(img)
                d.text( (2,2), str(idx), fill=255)
                b = np.asarray(img)
                video_writer.append_data(b)
                idx+=1
            video_writer.close()
            Video(video_path, embed=True)
            vidcount+=1

data.close()

# Make TRAIN TEST splits based on Objects

In [171]:
data['data']['demo_0']['action_dict'].keys()


<HDF5 dataset "actions": shape (246, 12), type "<f8">

In [46]:
''' CREATE 1 DATASET '''
def create_dataset_split(current_dataset, object_list, split, base_path):
    savepath = base_path+f'/demo_gentex_im128_randcams_new_images_{split}.hdf5'
    f = h5py.File(savepath, 'w')
    datagrp = f.create_group('data')
    datagrp.attrs['ogdataset'] = base_path
    datagrp.attrs['env_args'] = current_dataset['data'].attrs['env_args']
    # total=0
    # for i in paths:
    #     temp = h5py.File(i,'r')
    #     total+=temp['data'].attrs['total']
    # datagrp.attrs['total']=total

    count=0
    for demo in current_dataset['data']:
        object_is_in_list = False
        for objs in json.loads(current_dataset['data'][demo].attrs['ep_meta'])['object_cfgs']:
            if objs['name']=='obj':
                if objs['info']['cat'] in object_list:
                    # print('OBJECT FOUND', objs['info']['cat'])
                    object_is_in_list=True
        if object_is_in_list:
            demogrp = datagrp.create_group('demo_'+str(count))
            for k in current_dataset['data'][demo].attrs.keys():
                demogrp.attrs[k] = current_dataset['data'][demo].attrs[k]
            demogrp.attrs['og_demo_id']=demo
            actionsdset = demogrp.create_dataset('actions', data = current_dataset['data'][demo]['actions'])
            rewardsdset = demogrp.create_dataset('rewards', data = current_dataset['data'][demo]['rewards'])
            statesdset = demogrp.create_dataset('states', data = current_dataset['data'][demo]['states'])
            donesdset = demogrp.create_dataset('dones', data = current_dataset['data'][demo]['dones'])
            obsgrp = demogrp.create_group('obs') 
            for grp_name in current_dataset['data'][demo]['obs']:
                dset = obsgrp.create_dataset(grp_name, data = current_dataset['data'][demo]['obs'][grp_name])
            actiondictgrp = demogrp.create_group('action_dict') 
            for grp_name in current_dataset['data'][demo]['action_dict']:
                dset = actiondictgrp.create_dataset(grp_name, data = current_dataset['data'][demo]['action_dict'][grp_name])
            print('demo done', demo, count)
            count += 1

    print('dataset done', savepath)
    f.close()


In [53]:
# Open HDF5 file and write in the data_dict structure and info
base_path = '/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPCounterToStove/2024-04-26'
all_objects={}
data=h5py.File(base_path+'/demo_gentex_im128_randcams_new_images.hdf5', 'r')
for x in data['data']:
    try:
        for objs in json.loads(data['data'][x].attrs['ep_meta'])['object_cfgs']:
            if objs['name']=='obj':
                all_objects[objs['info']['cat']] = all_objects.get(objs['info']['cat'],0) + 1
    except:
        print('issue:', root, x)

all_objects = dict(sorted(all_objects.items(), key=lambda item: item[1]))
print(json.dumps(all_objects, indent=4))
val_objects = list(all_objects.keys())[:5]
train_objects = list(all_objects.keys())[5:]
print('val objects', val_objects)
print('train objects', train_objects)
create_dataset_split(data, train_objects, 'train', base_path)
create_dataset_split(data, val_objects, 'val', base_path)

{
    "steak": 1,
    "egg": 1,
    "lemon": 1,
    "eggplant": 1,
    "mango": 2,
    "tomato": 2,
    "garlic": 2,
    "onion": 2,
    "bell_pepper": 2,
    "fish": 2,
    "broccoli": 3,
    "sweet_potato": 3,
    "apple": 3,
    "carrot": 3,
    "potato": 4,
    "lime": 4,
    "cheese": 5,
    "corn": 10
}
val objects ['steak', 'egg', 'lemon', 'eggplant', 'mango']
train objects ['tomato', 'garlic', 'onion', 'bell_pepper', 'fish', 'broccoli', 'sweet_potato', 'apple', 'carrot', 'potato', 'lime', 'cheese', 'corn']
demo done demo_0 0
demo done demo_1 1
demo done demo_10 2
demo done demo_11 3
demo done demo_12 4
demo done demo_13 5
demo done demo_15 6
demo done demo_17 7
demo done demo_18 8
demo done demo_19 9
demo done demo_2 10
demo done demo_20 11
demo done demo_21 12
demo done demo_22 13
demo done demo_24 14
demo done demo_26 15
demo done demo_27 16
demo done demo_28 17
demo done demo_29 18
demo done demo_3 19
demo done demo_30 20
demo done demo_31 21
demo done demo_32 22
demo done d

# Combine Datasets

# These are the tasks we are combingin into 1 dataset:


In [57]:
paths_x_to_counter = [
        '/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPSinkToCounter/2024-04-26_2/demo_gentex_im128_randcams_new_images_val.hdf5',
         '/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPStoveToCounter/2024-05-01/demo_gentex_im128_randcams_new_images_val.hdf5',
         '/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPMicrowaveToCounter/2024-04-26/demo_gentex_im128_randcams_new_images_val.hdf5',
         '/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPCabToCounter/2024-04-24/demo_gentex_im128_randcams_new_images_val.hdf5'
]
paths_counter_to_x = [
    '/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPCounterToMicrowave/2024-04-27/demo_gentex_im128_randcams_new_images_val.hdf5',
    '/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPCounterToSink/2024-04-25/demo_gentex_im128_randcams_new_images_val.hdf5',
    '/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPCounterToStove/2024-04-26/demo_gentex_im128_randcams_new_images_val.hdf5',
    '/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPCounterToCab/2024-04-24/demo_gentex_im128_randcams_new_images_val.hdf5'
]
paths=paths_x_to_counter+paths_counter_to_x
print(*paths, sep="\n")

/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPSinkToCounter/2024-04-26_2/demo_gentex_im128_randcams_new_images_val.hdf5
/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPStoveToCounter/2024-05-01/demo_gentex_im128_randcams_new_images_val.hdf5
/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPMicrowaveToCounter/2024-04-26/demo_gentex_im128_randcams_new_images_val.hdf5
/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPCabToCounter/2024-04-24/demo_gentex_im128_randcams_new_images_val.hdf5
/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPCounterToMicrowave/2024-04-27/demo_gentex_im128_randcams_new_images_val.hdf5
/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPCounterToSink/2024-04-25/demo_gentex_im128_randcams_new_images_val.hdf5
/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitc

In [58]:
''' CREATE 1 DATASET '''
# Open HDF5 file and write in the data_dict structure and info
base_path = '/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/combined/PnPX'
savepath = base_path+'/demo_gentex_im128_randcams_new_images_val.hdf5'
testdata = h5py.File(base_path+'/demo_gentex_im128_randcams_new_images.hdf5', 'r')
f = h5py.File(savepath, 'w')
datagrp = f.create_group('data')
datagrp.attrs['datasets_included'] = ', '.join(paths)
datagrp.attrs['env_args'] = testdata['data'].attrs['env_args']
# total=0

# for i in paths:
#     temp = h5py.File(i,'r')
#     total+=temp['data'].attrs['total']
# datagrp.attrs['total']=total

count=0
for current_path in paths:
    current_dataset = h5py.File(current_path)
    for demo in current_dataset['data']:
        demogrp = datagrp.create_group('demo_'+str(count))
        for k in current_dataset['data'][demo].attrs.keys():
            demogrp.attrs[k] = current_dataset['data'][demo].attrs[k]
        demogrp.attrs['og_demo_id']=demo
        demogrp.attrs['dataset_path']=current_path
        actionsdset = demogrp.create_dataset('actions', data = current_dataset['data'][demo]['actions'])
        rewardsdset = demogrp.create_dataset('rewards', data = current_dataset['data'][demo]['rewards'])
        statesdset = demogrp.create_dataset('states', data = current_dataset['data'][demo]['states'])
        donesdset = demogrp.create_dataset('dones', data = current_dataset['data'][demo]['dones'])
        obsgrp = demogrp.create_group('obs') 
        for grp_name in current_dataset['data'][demo]['obs']:
            dset = obsgrp.create_dataset(grp_name, data = current_dataset['data'][demo]['obs'][grp_name])
        actiondictgrp = demogrp.create_group('action_dict') 
        for grp_name in current_dataset['data'][demo]['action_dict']:
            dset = actiondictgrp.create_dataset(grp_name, data = current_dataset['data'][demo]['action_dict'][grp_name])
        print('demo done', count)
        count += 1

    print('dataset done', current_path)
f.close()

demo done 0
demo done 1
demo done 2
demo done 3
demo done 4
dataset done /proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPSinkToCounter/2024-04-26_2/demo_gentex_im128_randcams_new_images_val.hdf5
demo done 5
demo done 6
demo done 7
demo done 8
demo done 9
demo done 10
dataset done /proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPStoveToCounter/2024-05-01/demo_gentex_im128_randcams_new_images_val.hdf5
demo done 11
demo done 12
demo done 13
demo done 14
demo done 15
demo done 16
demo done 17
dataset done /proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPMicrowaveToCounter/2024-04-26/demo_gentex_im128_randcams_new_images_val.hdf5
demo done 18
demo done 19
demo done 20
demo done 21
demo done 22
dataset done /proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPCabToCounter/2024-04-24/demo_gentex_im128_randcams_new_images_val.hdf5
demo done 23
demo done 24
demo done 25
d

In [140]:
'''TEST THE NEW DATASET'''

data = h5py.File(savepath, 'r')
print(data['data'])
video_path = base_path+'temp1.mp4'
video_writer = imageio.get_writer(video_path, fps=20)
idx = 6040
demo = f'demo_{idx}'
print(data['data'][demo]['object'])
print(data['data'][demo]['obs']['agentview_image'].shape)
print(data['data'][demo]['success'][:])
for b in  data['data'][demo]['obs']['agentview_image']:
    img = Image.fromarray((b).astype(np.uint8))
    d = ImageDraw.Draw(img)
    d.text( (2,2), str(idx), fill=255)
    
    
    #-- back to array
    b = np.asarray(img)

    video_writer.append_data(b)
    idx+=1
video_writer.close()
Video(video_path, embed=True)
data.close()

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (84, 84) to (96, 96) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


<HDF5 group "/data" (6048 members)>
<HDF5 dataset "object": shape (), type "|O">
(100, 84, 84, 3)
[0.]


[swscaler @ 0x569b080] Warning: data is not aligned! This can lead to a speed loss
